# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [19]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5
%additional_python_modules matplotlib,ipython

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session e7a91542-02d7-4158-b999-0c8324e64f2d.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is 2880 minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session e7a91542-02d7-4158-b999-0c8324e64f2d.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 5.1


You are already connected to a glueetl session e7a91542-02d7-4158-b999-0c8324e64f2d.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: G.1X
Setting new worker type to: G.1X


You are already connected to a glueetl session e7a91542-02d7-4158-b999-0c8324e64f2d.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: 5
Setting new number of workers to: 5


You are already connected to a glueetl session e7a91542-02d7-4158-b999-0c8324e64f2d.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Additional python modules to be included:
matplotlib
ipython



#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [ ]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='database_name', table_name='table_name')
dyf.printSchema()

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [ ]:
df = dyf.toDF()
df.show()

#### Example: Visualize data with matplotlib


In [ ]:
import matplotlib.pyplot as plt

# Set X-axis and Y-axis values
x = [5, 2, 8, 4, 9]
y = [10, 4, 8, 5, 2]
  
# Create a bar chart 
plt.bar(x, y)
  
# Show the plot
%matplot plt

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [ ]:
s3output = glueContext.getSink(
  path="s3://bucket_name/folder_name",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="demo", catalogTableName="populations"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(DyF)

In [4]:
# Teste de conexão com o Data Catalog

spark.sql("""
SHOW TABLES IN state_of_data
""").show(truncate=False)

+-------------+----------------+-----------+
|namespace    |tableName       |isTemporary|
+-------------+----------------+-----------+
|state_of_data|gold_2023       |false      |
|state_of_data|gold_2024       |false      |
|state_of_data|gold_2025       |false      |
|state_of_data|gold_consolidada|false      |
|state_of_data|tb_2023         |false      |
|state_of_data|tb_2024         |false      |
|state_of_data|tb_2025         |false      |
+-------------+----------------+-----------+


In [5]:
# Teste da tabela Gold

spark.sql("""
SELECT COUNT(*) AS total_registros
FROM state_of_data.gold_consolidada
""").show()

+---------------+
|total_registros|
+---------------+
|          14005|
+---------------+


In [6]:
# Análise de senioridade

df_nivel = spark.sql("""
SELECT
    ano,
    nivel,
    COUNT(*) AS quantidade,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (PARTITION BY ano),
        2
    ) AS percentual
FROM state_of_data.gold_consolidada
WHERE nivel IS NOT NULL
  AND TRIM(nivel) <> ''
GROUP BY ano, nivel
ORDER BY ano, quantidade DESC
""")

df_nivel.show(20, truncate=False)

+----+-------------------+----------+----------+
|ano |nivel              |quantidade|percentual|
+----+-------------------+----------+----------+
|2023|Sênior             |1419      |36.79     |
|2023|Pleno              |1392      |36.09     |
|2023|Júnior             |1046      |27.12     |
|2024|Sênior             |1573      |41.20     |
|2024|Pleno              |1377      |36.07     |
|2024|Júnior             |868       |22.73     |
|2025|Sênior             |858       |34.31     |
|2025|Pleno              |776       |31.03     |
|2025|Júnior             |518       |20.71     |
|2025|Especialista/Staff+|349       |13.95     |
+----+-------------------+----------+----------+


In [7]:
# Pergunta 1 - Estrutura do mercado
# Análise dos principais cargos

df_cargo = spark.sql("""
SELECT
    ano,
    cargo,
    COUNT(*) AS quantidade,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (PARTITION BY ano),
        2
    ) AS percentual
FROM state_of_data.gold_consolidada
WHERE cargo IS NOT NULL
  AND TRIM(cargo) <> ''
GROUP BY ano, cargo
ORDER BY ano, quantidade DESC
""")

df_cargo.show(50, truncate=False)

+----+-------------------------------------------------------------------+----------+----------+
|ano |cargo                                                              |quantidade|percentual|
+----+-------------------------------------------------------------------+----------+----------+
|2023|Analista de Dados/Data Analyst                                     |907       |23.52     |
|2023|Cientista de Dados/Data Scientist                                  |687       |17.81     |
|2023|Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect|684       |17.73     |
|2023|Analista de BI/BI Analyst                                          |506       |13.12     |
|2023|Outra Opção                                                        |262       |6.79      |
|2023|Analista de Negócios/Business Analyst                              |195       |5.06      |
|2023|Analytics Engineer                                                 |139       |3.60      |
|2023|Desenvolvedor/ Engenheir

In [8]:
df_setor = spark.sql("""
SELECT
    ano,
    setor,
    COUNT(*) AS quantidade,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (PARTITION BY ano),
        2
    ) AS percentual
FROM state_of_data.gold_consolidada
WHERE setor IS NOT NULL
  AND TRIM(setor) <> ''
GROUP BY ano, setor
ORDER BY ano, quantidade DESC
""")

df_setor.show(70, truncate=False)

+----+-----------------------------------+----------+----------+
|ano |setor                              |quantidade|percentual|
+----+-----------------------------------+----------+----------+
|2023|Finanças ou Bancos                 |927       |19.50     |
|2023|Tecnologia/Fábrica de Software     |857       |18.03     |
|2023|Varejo                             |392       |8.25      |
|2023|Área de Consultoria                |389       |8.18      |
|2023|Outra Opção                        |344       |7.24      |
|2023|Indústria                          |313       |6.59      |
|2023|Educação                           |209       |4.40      |
|2023|Área da Saúde                      |201       |4.23      |
|2023|Setor Público                      |174       |3.66      |
|2023|Internet/Ecommerce                 |129       |2.71      |
|2023|Marketing                          |115       |2.42      |
|2023|Telecomunicação                    |115       |2.42      |
|2023|Setor Alimentício  

In [20]:
import matplotlib.pyplot as plt

print("Matplotlib:", plt.matplotlib.__version__)

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    ["2023", "2024", "2025"],
    [27.12, 22.73, 20.71]
)

ax.set_title("Teste de visualização")
ax.set_ylabel("Percentual (%)")

plt.tight_layout()
plt.show()

Matplotlib: 3.10.7


In [24]:
# ============================================================
# GRÁFICO 1 - SENIORIDADE
# Salvar diretamente no S3
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Converter Spark -> Pandas
# ------------------------------------------------------------

nivel_pd = df_nivel.toPandas()

# Corrigir tipo Decimal -> float
nivel_pd["percentual"] = nivel_pd["percentual"].astype(float)

anos = [2023, 2024, 2025]

ordem_nivel = [
    "Sênior",
    "Pleno",
    "Júnior",
    "Especialista/Staff+"
]

# ------------------------------------------------------------
# Preparar dados
# ------------------------------------------------------------

base_nivel = (
    nivel_pd
    .pivot(
        index="ano",
        columns="nivel",
        values="percentual"
    )
    .reindex(anos)
    .fillna(0)
)

for nivel in ordem_nivel:
    if nivel not in base_nivel.columns:
        base_nivel[nivel] = 0.0

base_nivel = base_nivel[ordem_nivel].astype(float)

# ------------------------------------------------------------
# Criar gráfico
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 7))

bottom = np.zeros(len(anos), dtype=float)

cores = {
    "Sênior": "#123B63",
    "Pleno": "#2F80C0",
    "Júnior": "#8FC4E8",
    "Especialista/Staff+": "#F2994A"
}

for nivel in ordem_nivel:

    valores = base_nivel[nivel].to_numpy(dtype=float)

    ax.bar(
        anos,
        valores,
        bottom=bottom,
        label=nivel,
        color=cores[nivel]
    )

    for i, valor in enumerate(valores):

        if valor >= 5:

            ax.text(
                anos[i],
                bottom[i] + valor / 2,
                f"{valor:.1f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=12,
                fontweight="bold"
            )

    bottom += valores

# ------------------------------------------------------------
# Formatação
# ------------------------------------------------------------

ax.set_title(
    "Distribuição dos profissionais por senioridade",
    fontsize=19,
    fontweight="bold",
    pad=15
)

ax.set_ylabel(
    "Percentual dos respondentes",
    fontsize=12
)

ax.set_xlabel(
    "Ano da pesquisa",
    fontsize=12
)

ax.set_ylim(0, 100)

ax.set_xticks(anos)

ax.grid(
    axis="y",
    alpha=0.25
)

ax.set_axisbelow(True)

ax.legend(
    title="Senioridade",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.10),
    ncol=4,
    frameon=False
)

plt.tight_layout()

# ------------------------------------------------------------
# Salvar localmente
# ------------------------------------------------------------

arquivo = "/tmp/grafico_senioridade.png"

plt.savefig(
    arquivo,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

print("Gráfico criado com sucesso:")
print(arquivo)

Gráfico criado com sucesso:
/tmp/grafico_senioridade.png


In [25]:
# ============================================================
# ENVIAR GRÁFICO PARA O S3
# ============================================================

import boto3

s3 = boto3.client("s3")

bucket = "lab-385615870279"

destino = "data-output/graficos/pergunta_1/grafico_senioridade.png"

s3.upload_file(
    "/tmp/grafico_senioridade.png",
    bucket,
    destino
)

print("Upload concluído!")
print(f"s3://{bucket}/{destino}")

Upload concluído!
s3://lab-385615870279/data-output/graficos/pergunta_1/grafico_senioridade.png
